In [3]:
import math
import random

# Генерація гіпотетичної вибірки зображень руки
# Формат: (Висота, Ширина, Клас)
# Клас 0: Кулак (мала висота, мала ширина)
# Клас 1: Долоня (велика висота, велика ширина)
random.seed(42)
data_fist = [(random.uniform(5, 8), random.uniform(5, 8), 0) for _ in range(20)]
data_palm = [(random.uniform(12, 18), random.uniform(10, 16), 1) for _ in range(20)]
dataset = data_fist + data_palm

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# Класифікація за алгоритмом k-найближчих сусідів
def knn_classify(train_data, test_point, k=3):
    distances = []
    for row in train_data:
        dist = euclidean_distance((row[0], row[1]), test_point)
        distances.append((dist, row[2]))

    # Сортуємо за відстанню і беремо k найближчих
    distances.sort(key=lambda x: x[0])
    neighbors = distances[:k]

    # Визначаємо найчастіший клас
    classes = [neighbor[1] for neighbor in neighbors]
    prediction = max(set(classes), key=classes.count)
    return prediction

# Кластеризація за алгоритмом k-середніх
def kmeans_cluster(data, k, max_iters=100):
    points = [(row[0], row[1]) for row in data]
    # Початкові центроїди випадковим чином
    centroids = random.sample(points, k)

    for _ in range(max_iters):
        clusters = [[] for _ in range(k)]

        # Розподіляємо точки по найближчих центроїдах
        for point in points:
            distances = [euclidean_distance(point, c) for c in centroids]
            closest_idx = distances.index(min(distances))
            clusters[closest_idx].append(point)

        # Перераховуємо центри
        new_centroids = []
        for cluster in clusters:
            if not cluster:
                # Якщо кластер порожній, залишаємо випадкову точку
                new_centroids.append(random.choice(points))
                continue
            avg_h = sum(p[0] for p in cluster) / len(cluster)
            avg_w = sum(p[1] for p in cluster) / len(cluster)
            new_centroids.append((avg_h, avg_w))

        # Якщо центроїди не змінилися, зупиняємось
        if centroids == new_centroids:
            break
        centroids = new_centroids

    return clusters, centroids

# Демонстрація роботи алгоритмів
print(" Тестування k-NN ")
test_point_1 = (6.5, 6.0) # Схоже на кулак
test_point_2 = (15.0, 14.0) # Схоже на долоню
print(f"Точка {test_point_1} класифікована як: {'Кулак' if knn_classify(dataset, test_point_1) == 0 else 'Долоня'}")
print(f"Точка {test_point_2} класифікована як: {'Кулак' if knn_classify(dataset, test_point_2) == 0 else 'Долоня'}")

print("\n Тестування k-means ")
for k_val in [2, 3, 5]:
    clusters, centroids = kmeans_cluster(dataset, k_val)
    print(f"\nРозбиття на k={k_val} кластерів:")
    for i, cluster in enumerate(clusters):
        print(f"  Кластер {i+1} (Центр: {centroids[i][0]:.2f}, {centroids[i][1]:.2f}): {len(cluster)} точок")

 Тестування k-NN 
Точка (6.5, 6.0) класифікована як: Кулак
Точка (15.0, 14.0) класифікована як: Долоня

 Тестування k-means 

Розбиття на k=2 кластерів:
  Кластер 1 (Центр: 14.77, 12.65): 20 точок
  Кластер 2 (Центр: 6.70, 6.23): 20 точок

Розбиття на k=3 кластерів:
  Кластер 1 (Центр: 6.70, 6.23): 20 точок
  Кластер 2 (Центр: 14.31, 11.53): 12 точок
  Кластер 3 (Центр: 15.45, 14.33): 8 точок

Розбиття на k=5 кластерів:
  Кластер 1 (Центр: 16.59, 13.18): 8 точок
  Кластер 2 (Центр: 13.64, 11.32): 9 точок
  Кластер 3 (Центр: 13.30, 15.22): 3 точок
  Кластер 4 (Центр: 7.38, 6.60): 11 точок
  Кластер 5 (Центр: 5.87, 5.79): 9 точок
